In [ ]:
!pip install faker -q
import pandas as pd
import numpy as np
import json
import time
from concurrent.futures import ThreadPoolExecutor, as_completed
from faker import Faker
import random

In [ ]:
data = pd.DataFrame({'x': range(1,5)})
def my_func(x):
    time.sleep(.8)
    if x==2:
        if count != data.shape[0]-1:
            raise Exception("This is an error")
    return x**2
results = []
count = 0
with ThreadPoolExecutor(max_workers=2) as executor:
    futures = {executor.submit(my_func, row['x']): index for index, row in data.iterrows()}

    # Collect results as tasks complete
    for future in as_completed(futures):
        index = futures[future]  # Get the original index
        try:
            result = future.result()  # Get the result of the completed task
            count += 1
            results.append({'index': index, 'square': result})
        except Exception as e:
            print(e)
        if count == data.shape[0]:
            break
    print('Complete')

df_results = data.join(pd.DataFrame(results).set_index('index'))
print("\nProcessed Sentences:")
display(df_results)

This is an error
Complete

Processed Sentences:


,x,square
0,1,1.0
1,2,NaN
2,3,9.0
3,4,16.0


In [ ]:
def run_with_retry(task_func, *args, retries=3):
    for attempt in range(retries):
        try:
            return task_func(*args)  # Try running the task
        except Exception as e:
            print(f"Attempt {attempt + 1} failed with error: {e}")
            if attempt == retries - 1:  # If last attempt fails, re-raise the exception
                raise
            time.sleep(1)  # Optional: Wait before retrying

In [ ]:
from concurrent.futures import ThreadPoolExecutor, as_completed
import pandas as pd
from IPython.display import display
import time

data = pd.DataFrame({'x': range(1,5)})
def my_func(x):
    time.sleep(2)
    if x==1:
        if count != data.shape[0]-1:
            raise Exception(f"This is an error processing the {x} value")
    return x**2
results = []
count = 0
with ThreadPoolExecutor(max_workers=2) as executor:
    futures = {executor.submit(run_with_retry,my_func, row['x']): index for index, row in data.iterrows()}

    for future in as_completed(futures):
        index = futures[future]  # Get the original index
        try:
            result = future.result()  # Get the result of the completed task
            count += 1
            results.append({'index': index, 'square': result})
        except Exception as e:
            print(e)
        if count == data.shape[0]:
            break

# Create a DataFrame from the results
df_results = data.join(pd.DataFrame(results).set_index('index'))

print("\nProcessed Sentences:")
display(df_results)

Attempt 1 failed with error: This is an error processing the 1 value
Attempt 2 failed with error: This is an error processing the 1 value

Processed Sentences:


,x,square
0,1,1
1,2,4
2,3,9
3,4,16


In [ ]:
from openai import AzureOpenAI
from google.colab import userdata
client = AzureOpenAI(api_key=userdata.get('AZURE_API_KEY'), azure_endpoint=userdata.get('AZURE_BASE_URL'), api_version=userdata.get('AZURE_API_VERSION'))

In [ ]:
# Simulate a time-consuming function to process each row
def process_row(row):
    time.sleep(random.choice([3,2,4,5,1]))  # Simulating a delay to mimic a time-consuming operation
    return row['id'], row['value'] * 2  # Return the ID and double the value

In [ ]:
# Create a sample DataFrame
data = {
    'id': [1, 2, 3, 4, 5],
    'value': [10, 20, 30, 40, 50]
}
df = pd.DataFrame(data)

# List to hold results
results = []

# Using ThreadPoolExecutor to process rows in parallel
with ThreadPoolExecutor(max_workers=10) as executor:  # Limit to 3 concurrent threads
    # Submitting tasks for each row
    futures = {executor.submit(process_row, row): index for index, row in df.iterrows()}

    # Collect results as tasks complete
    for future in as_completed(futures):
        index = futures[future]  # Get the original index
        result = future.result()  # Get the result of the completed task
        results.append({'id': result[0], 'processed_value': result[1]})
        print("-"*50)
        display(results)
    print('Complete')

# Create a DataFrame from the results
df_results = pd.DataFrame(results)
print("Processed DataFrame:")
print(df_results)

--------------------------------------------------


[{'id': 3, 'processed_value': 60}]

--------------------------------------------------


[{'id': 3, 'processed_value': 60}, {'id': 4, 'processed_value': 80}]

--------------------------------------------------


...

--------------------------------------------------


[{'id': 3, 'processed_value': 60},
 {'id': 4, 'processed_value': 80},
 {'id': 5, 'processed_value': 100},
 {'id': 1, 'processed_value': 20}]

--------------------------------------------------


[{'id': 3, 'processed_value': 60},
 {'id': 4, 'processed_value': 80},
 {'id': 5, 'processed_value': 100},
 {'id': 1, 'processed_value': 20},
 {'id': 2, 'processed_value': 40}]

Complete
Processed DataFrame:
   id  processed_value
0   3               60
1   4               80
2   5              100
3   1               20
4   2               40


In [ ]:
def generate_sentence(user_name, user_country, user_language):
    # Construct the prompt
    prompt = f"Create a simple sentence using the user's name '{user_name}', country '{user_country}', and language '{user_language}' doing some action in the correct language, then translate to english."

    # Call the OpenAI API
    openai_response = client.chat.completions.create(
        model="gpt-4o-2",
        response_format={"type": "json_object"},
        messages=[
            {"role": "system", "content": "You are a helpful assistant which return a JSON blob with just two(2) key: response, translation."},
            {"role": "user", "content": prompt}
        ],
        temperature=0.1,
        max_tokens=50  # Limit the length of the response
    )
    response_content = openai_response.choices[0].message.content
    try:
        json_response = json.loads(response_content)
        return json_response['response'], json_response['translation']
    except json.JSONDecodeError:
        return response_content, 'NONE'

In [ ]:
fake = Faker()
dir(fake)

['__annotations__',
 '__class__',
 '__deepcopy__',
 '__delattr__',
 '__dict__',
 '__dir__',
 '__doc__',
 '__eq__',
 '__format__',
 '__ge__',
 '__getattr__',
 '__getattribute__',
 '__getitem__',
 '__gt__',
 '__hash__',
 '__init__',
 '__init_subclass__',
 '__le__',
 '__lt__',
 '__module__',
 '__ne__',
 '__new__',
 '__reduce__',
 '__reduce_ex__',
 '__repr__',
 '__setattr__',
 '__setstate__',
 '__sizeof__',
 '__str__',
 '__subclasshook__',
 '__weakref__',
 '_factories',
 '_factory_map',
 '_locales',
 '_map_provider_method',
 '_optional_proxy',
 '_select_factory',
 '_select_factory_choice',
 '_select_factory_distribution',
 '_unique_proxy',
 '_weights',
 'aba',
 'add_provider',
 'address',
 'administrative_unit',
 'am_pm',
 'android_platform_token',
 'ascii_company_email',
 'ascii_email',
 'ascii_free_email',
 'ascii_safe_email',
 'bank_country',
 'basic_phone_number',
 'bban',
 'binary',
 'boolean',
 'bothify',
 'bs',
 'building_number',
 'cache_pattern',
 'catch_phrase',
 'century',
 'chr

In [ ]:
def create_fake_dataframe(num_rows=10):
    fake = Faker()
    return pd.DataFrame({
        'name': [fake.name() for _ in range(num_rows)],
        'country': np.random.choice([fake.country() for _ in range(num_rows)], num_rows),
        'language': np.random.choice([fake.language_name() for _ in range(num_rows)], num_rows)
    })

In [ ]:
df = create_fake_dataframe(20)
print("Original DataFrame:")
display(df)

# List to hold results
results = []

# Using ThreadPoolExecutor to process rows in parallel
with ThreadPoolExecutor(max_workers=5) as executor:  # Limit to 10 concurrent threads
    # Submitting tasks for each row
    futures = {executor.submit(generate_sentence, row['name'], row['country'], row['language']): index for index, row in df.iterrows()}

    # Collect results as tasks complete
    for future in as_completed(futures):
        index = futures[future]  # Get the original index
        result, translation = future.result()  # Get the result of the completed task
        results.append({'index': index, 'sentence': result, 'ENG-translation': translation})

# Create a DataFrame from the results
df_results = pd.DataFrame(results)
print("\nProcessed Sentences:")
display(df_results)

final_df = df.join(df_results.set_index('index'))
final_df

Original DataFrame:


,name,country,language
0,Aaron Washington,South Georgia and the South Sandwich Islands,Hausa
1,Michael Cunningham,Oman,Czech
2,Russell Ward,Chile,Czech
3,Samantha Turner,Kyrgyz Republic,Romansh
4,Benjamin Moon,Chile,Czech
5,Shelby Park,Oman,Sichuan Yi
6,Sarah Simpson,South Georgia and the South Sandwich Islands,Hausa
7,Samantha Herrera,Norfolk Island,Malayalam
8,Garrett James,South Georgia and the South Sandwich Islands,Shona
9,Katelyn Warner,Rwanda,Hausa



Processed Sentences:


,index,sentence,ENG-translation
0,2,Russell Ward cestuje do Chile.,Russell Ward travels to Chile.
1,0,Aaron Washington yana kallon teku a Kudancin G...,Aaron Washington is watching the sea in South ...
2,1,Michael Cunningham mluví česky v Ománu.,Michael Cunningham speaks Czech in Oman.
3,7,സമന്ത ഹെറേര നോർഫോക്ക് ദ്വീപിൽ മലയാളം സംസാരിക്ക...,Samantha Herrera is speaking Malayalam in Norf...
4,5,Shelby Park 在阿曼用四川彝语唱歌。,Shelby Park is singing in Sichuan Yi in Oman.
5,4,Benjamin Moon cestuje do Chile a učí se česky.,Benjamin Moon travels to Chile and learns Czech.
6,9,Katelyn Warner tana koyon Hausa a Rwanda.,Katelyn Warner is learning Hausa in Rwanda.
7,8,Garrett James ari kutamba muSouth Georgia neSo...,Garrett James is playing in South Georgia and ...
8,10,"{\n ""response"": ""Lisa Mcdonald hè in British ...",NONE
9,12,ബ്രയാൻ സ്റ്റീൽ സെന്റ് ലൂസിയയിൽ മലയാളം പഠിക്കുന...,Brian Steele is learning Malayalam in Saint Lu...


,name,country,language,sentence,ENG-translation
0,Aaron Washington,South Georgia and the South Sandwich Islands,Hausa,Aaron Washington yana kallon teku a Kudancin G...,Aaron Washington is watching the sea in South ...
1,Michael Cunningham,Oman,Czech,Michael Cunningham mluví česky v Ománu.,Michael Cunningham speaks Czech in Oman.
2,Russell Ward,Chile,Czech,Russell Ward cestuje do Chile.,Russell Ward travels to Chile.
3,Samantha Turner,Kyrgyz Republic,Romansh,Samantha Turner gioga en il Kyrgyz Republic.,Samantha Turner is playing in the Kyrgyz Repub...
4,Benjamin Moon,Chile,Czech,Benjamin Moon cestuje do Chile a učí se česky.,Benjamin Moon travels to Chile and learns Czech.
5,Shelby Park,Oman,Sichuan Yi,Shelby Park 在阿曼用四川彝语唱歌。,Shelby Park is singing in Sichuan Yi in Oman.
6,Sarah Simpson,South Georgia and the South Sandwich Islands,Hausa,Sarah Simpson tana yin aiki a South Georgia da...,Sarah Simpson is working in South Georgia and ...
7,Samantha Herrera,Norfolk Island,Malayalam,സമന്ത ഹെറേര നോർഫോക്ക് ദ്വീപിൽ മലയാളം സംസാരിക്ക...,Samantha Herrera is speaking Malayalam in Norf...
8,Garrett James,South Georgia and the South Sandwich Islands,Shona,Garrett James ari kutamba muSouth Georgia neSo...,Garrett James is playing in South Georgia and ...
9,Katelyn Warner,Rwanda,Hausa,Katelyn Warner tana koyon Hausa a Rwanda.,Katelyn Warner is learning Hausa in Rwanda.


In [ ]:
# prompt: select rows with eng translation column in "NONE" value and print sentence column

# Assuming 'final_df' is the DataFrame from the previous code
none_eng_sentences = final_df[final_df['ENG-translation'] == 'NONE']['sentence']
display(none_eng_sentences)
none_eng_sentences.shape[0]

,sentence
5,"{\n ""response"": ""Ο Κρίστοφερ Γκονζάλες μαθαίν..."
11,"{\n ""response"": ""Arthur Gonzales ꊇꉙꌠꉙꂷꀋꋍ Nort..."
19,"{\n ""response"": ""ᔭᐃᒥᔅ ᐅᒃᑲᖑᖅ ᐊᓐᑑᕋ"


3

In [ ]:
from pydantic import BaseModel, Field

class LanguageResponse(BaseModel):
    sentence: str = Field(...,description="A sentence generated in the indicated language")
    eng_translation: str = Field(...,description="English translation of the sentence")

def generate_sentence(user_name, user_country, user_language):
    prompt = f"Create a simple sentence using the user's name '{user_name}', country '{user_country}', and language '{user_language}' doing some action in the correct language, then translate to english."
    openai_response = client.beta.chat.completions.parse(
        model="gpt-4o-2",
        response_format=LanguageResponse,
        messages=[
            {"role": "system", "content": "You are a helpful assistant which return a JSON blob with just two(2) key: response, translation."},
            {"role": "user", "content": prompt}
        ],
        temperature=0.0,
        max_tokens=None
    )
    response = openai_response.choices[0].message.parsed
    return response.sentence, response.eng_translation

In [ ]:
generate_sentence("Freddy","Peru","Spanish")

('Freddy está visitando Perú.', 'Freddy is visiting Peru.')

In [ ]:
client.beta.chat.completions.parse

<bound method Completions.parse of <openai.resources.beta.chat.completions.Completions object at 0x7ff0f6930250>>

In [ ]:
import time
from tenacity import (
    retry,
    stop_after_attempt,
    wait_random_exponential,
)

@retry(wait=wait_random_exponential(min=1, max=3), stop=stop_after_attempt(6))
def generate_sentence(user_name, user_country, user_language):
    # time.sleep(2)
    prompt = f"Create a simple sentence using the user's name '{user_name}', country '{user_country}', and language '{user_language}' doing some action in the correct language, then translate to english."
    openai_response = client.beta.chat.completions.parse(
        model="gpt-4o-2",
        response_format=LanguageResponse,
        messages=[
            {"role": "system", "content": "You are a helpful assistant which return a JSON blob with just two(2) key: response, translation."},
            {"role": "user", "content": prompt}
        ],
        temperature=0.0,
        max_tokens=None
    )
    response = openai_response.choices[0].message.parsed
    return response.sentence, response.eng_translation

In [ ]:
generate_sentence("Freddy","Peru","Spanish")

('Freddy está visitando Perú.', 'Freddy is visiting Peru.')

In [ ]:
df = create_fake_dataframe(1000)
print("Original DataFrame:")
display(df.sample())

# List to hold results
results = []

# Using ThreadPoolExecutor to process rows in parallel
with ThreadPoolExecutor(max_workers=50) as executor:
    # Submitting tasks for each row
    futures = {executor.submit(generate_sentence, row['name'], row['country'], row['language']): index for index, row in df.iterrows()}

    # Collect results as tasks complete
    for future in as_completed(futures):
        index = futures[future]  # Get the original index
        try:
            result, translation = future.result()  # Get the result of the completed task
            results.append({'index': index, 'sentence': result, 'ENG-translation': translation})
            print(len(results))
        except Exception as e:
            print(e)

# Create a DataFrame from the results
df_results = pd.DataFrame(results)
print("\nProcessed Sentences:")
display(df_results.sample())

final_df = df.join(df_results.set_index('index'))
final_df.sample()

Original DataFrame:


,name,country,language
298,Mark Williams,Comoros,North Ndebele


1
2
3
4
5
6
7
8
9
10
11
12
13
14
15
16
17
18
19
20
21
22
23
24
25
26
27
28
29
30
31
32
33
34
35
36
37
38
39
40
41
42
43
44
45
46
47
48
49
50
51
52
53
54
55
56
57
58
59
60
61
62
63
64
65
66
67
68
69
70
71
72
73
74
75
76
77
78
79
80
81
82
83
84
85
86
87
88
89
90
91
92
93
94
95
96
97
98
99
100
101
102
103
104
105
106
107
108
109
110
111
112
113
114
115
116
117
118
119
120
121
122
123
124
125
126
127
128
129
130
131
132
133
134
135
136
137
138
139
140
141
142
143
144
145
146
147
148
149
150
151
152
153
154
155
156
157
158
159
160
161
162
163
164
165
166
167
168
169
170
171
172
173
174
175
176
177
178
179
180
181
182
183
184
185
186
187
188
189
190
191
192
193
194
195
196
197
198
199
200
201
202
203
204
205
206
207
208
209
210
211
212
213
214
215
216
217
218
219
220
221
222
223
224
225
226
227
228
229
230
231
232
233
234
235
236
237
238
239
240
241
242
243
244
245
246
247
248
249
250
251
252
253
254
255
256
257
258
259
260
261
262
263
264
265
266
267
268
269
270
271
272
273
274
275
276
277


For 20 rows, takes around 14 seconds using 5 workers

For 100 rows, takes around 14 seconds using 5 workers


In [ ]:
def emb(text:str):
    openai_response = client.embeddings.create(input=[text],  model="cwstxtemb3small"  )
    return openai_response.data[0].embedding

results = []
rows_simultaniously = 5

# Using ThreadPoolExecutor to process rows in parallel
with ThreadPoolExecutor(max_workers=50) as executor:  # Limit to 10 concurrent threads
    # Submitting tasks for each row
    futures = {executor.submit(emb, row['ENG-translation']): index for index, row in final_df.iterrows()}

    # Collect results as tasks complete
    for future in as_completed(futures):
        index = futures[future]
        embedding = future.result()
        results.append({'index': index, 'embedding': embedding})

# Create a DataFrame from the results
df_results = pd.DataFrame(results)
print("\nProcessed Sentences:")
display(df_results.sample())

final_df = final_df.join(df_results.set_index('index'))
final_df.sample()


Processed Sentences:


,index,embedding
1,10,"[-0.018207788467407227, 0.0498584620654583, -0..."


,name,country,language,sentence,ENG-translation,embedding
47,Amanda Gray,Armenia,Kinyarwanda,Amanda Gray arimo gusoma igitabo muri Armenia.,Amanda Gray is reading a book in Armenia.,"[-0.01012799609452486, 0.04527530074119568, 0...."


In [ ]:
final_df.shape

(50, 6)

In [ ]:
final_df.drop(['embedding'],axis=1,inplace=True,errors='ignore')

In [ ]:
def emb(texts: list):
    openai_response = client.embeddings.create(input=texts, model="cwstxtemb3small")
    return [response.embedding for response in openai_response.data]


results = []
rows_simultaniously = 5

# Using ThreadPoolExecutor to process rows in parallel
with ThreadPoolExecutor(max_workers=50) as executor:  # Limit to 50 concurrent threads
    futures = {}

    # Grouping rows into batches
    for index in range(0, len(final_df), rows_simultaniously):
        batch_rows = final_df.iloc[index:index + rows_simultaniously]
        texts = batch_rows['ENG-translation'].tolist()

        # Submitting a task for the batch of texts
        futures[executor.submit(emb, texts)] = batch_rows.index.tolist()

    # Collect results as tasks complete
    for future in as_completed(futures):
        indices = futures[future]
        embeddings = future.result()

        for idx, embedding in zip(indices, embeddings):
            results.append({'index': idx, 'embedding': embedding})

# Create a DataFrame from the results
df_results = pd.DataFrame(results)
print("\nProcessed Sentences:")
display(df_results.sample())

# Join the results back to the original DataFrame
final_df = final_df.join(df_results.set_index('index'))
final_df.head()


Processed Sentences:


,index,embedding
15,20,"[-0.0021259491331875324, 0.03220342472195625, ..."


,name,country,language,sentence,ENG-translation,embedding
0,Matthew Bruce,Namibia,Sindhi,میتھیو بروس نامیبیا ۾ ڪتاب پڙهي رهيو آهي.,Matthew Bruce is reading a book in Namibia.,"[-0.02359309419989586, 0.053618330508470535, 0..."
1,Richard Parker,Canada,Sardinian,Richard Parker est in Canada e leet unu libru.,Richard Parker is in Canada and is reading a b...,"[-0.004082714207470417, 0.02295321226119995, -..."
2,Taylor Lam,Libyan Arab Jamahiriya,Hebrew,טיילור לאם מבקר בלוב.,Taylor Lam is visiting Libya.,"[-0.004335521720349789, -0.015486391261219978,..."
3,Mr. Thomas Gillespie,French Polynesia,Greek,Ο κύριος Τόμας Γκιλέσπι επισκέπτεται τη Γαλλικ...,Mr. Thomas Gillespie visits French Polynesia.,"[-0.014608677476644516, 0.011110383085906506, ..."
4,Kevin Sanchez,Saint Vincent and the Grenadines,Hebrew,קווין סאנצ'ז מטייל בסנט וינסנט והגרנדינים.,Kevin Sanchez is traveling in Saint Vincent an...,"[0.005629299208521843, -0.00332862907089293, 0..."


In [ ]:
prompt = f"Create a simple sentence using the user's name 'Juan', country 'Colmbia', and language 'english' doing some action in the correct language, then translate to english."
openai_response = client.beta.chat.completions.parse(
    model="gpt-4o-2",
    response_format=LanguageResponse,
    messages=[
        {"role": "system", "content": "You are a helpful assistant which return a JSON blob with just two(2) key: response, translation."},
        {"role": "user", "content": prompt}
    ],
    temperature=0.0,
    max_tokens=None
)
response = openai_response.choices[0].message.parsed